# Orchestration with Airflow — prototyping the pipeline

Airflow DAGs run from a `dags/` folder read by the scheduler — not from notebooks.
The workflow here: prototype each task's logic in cells, then export it as a DAG file
(final section writes `../dags/etl_demo.py` for you).

## 1. Prototype the EXTRACT step

In [ ]:
import numpy as np
import pandas as pd

def extract(logical_date: str) -> pd.DataFrame:
    # Stand-in for an API call / file drop. Deterministic per date = idempotent.
    rng = np.random.default_rng(abs(hash(logical_date)) % (2**32))
    return pd.DataFrame({
        "order_id": range(1, 101),
        "amount": rng.uniform(5, 500, 100).round(2),
        "country": rng.choice(["ES", "FR", "DE", "US"], 100),
        "date": logical_date,
    })

raw = extract("2026-07-01")
raw.head(3)

## 2. Prototype the TRANSFORM step

In [ ]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    out = (df.groupby(["date", "country"], as_index=False)
             .agg(n_orders=("order_id", "count"),
                  revenue=("amount", "sum")))
    out["revenue"] = out["revenue"].round(2)
    return out

daily = transform(raw)
daily

## 3. Prototype the LOAD step (idempotent: delete-then-insert for the interval)

In [ ]:
import sqlite3

def load(df: pd.DataFrame, logical_date: str, db_path: str = "warehouse.db"):
    with sqlite3.connect(db_path) as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS daily_revenue
                        (date TEXT, country TEXT, n_orders INT, revenue REAL)""")
        # idempotency: a re-run for this date replaces, never duplicates
        conn.execute("DELETE FROM daily_revenue WHERE date = ?", (logical_date,))
        df.to_sql("daily_revenue", conn, if_exists="append", index=False)

load(daily, "2026-07-01")
load(daily, "2026-07-01")  # run twice on purpose — count must not double
with sqlite3.connect("warehouse.db") as conn:
    print(pd.read_sql("SELECT date, COUNT(*) n FROM daily_revenue GROUP BY date", conn))

## 4. Sketch the dependencies (what the DAG will encode)

In [ ]:
# extract >> transform >> load, with a file sensor gating extract.
# Fan-out example: one extract feeding two independent transforms would be:
#   extract >> [transform_daily, transform_by_country] >> load
for date in ["2026-07-01", "2026-07-02"]:   # manual 'backfill' of two intervals
    load(transform(extract(date)), date)
with sqlite3.connect("warehouse.db") as conn:
    print(pd.read_sql("SELECT * FROM daily_revenue ORDER BY date, country", conn).head(8))

## 5. Export as a real Airflow DAG (TaskFlow API)

In [ ]:
import os

DAG_CODE = '''
from datetime import datetime

import numpy as np
import pandas as pd
from airflow.decorators import dag, task


@dag(schedule="@daily", start_date=datetime(2026, 7, 1), catchup=False,
     default_args={"retries": 2})
def etl_demo():

    @task
    def extract(ds=None) -> str:
        rng = np.random.default_rng(abs(hash(ds)) % (2**32))
        df = pd.DataFrame({"order_id": range(1, 101),
                           "amount": rng.uniform(5, 500, 100).round(2),
                           "country": rng.choice(["ES", "FR", "DE", "US"], 100),
                           "date": ds})
        path = f"/tmp/raw_{ds}.parquet"      # pass a PATH through XCom, not the data
        df.to_parquet(path)
        return path

    @task
    def transform(path: str) -> str:
        df = pd.read_parquet(path)
        out = df.groupby(["date", "country"], as_index=False).agg(
            n_orders=("order_id", "count"), revenue=("amount", "sum"))
        out_path = path.replace("raw_", "daily_")
        out.to_parquet(out_path)
        return out_path

    @task
    def load(path: str, ds=None):
        import sqlite3
        df = pd.read_parquet(path)
        with sqlite3.connect("/tmp/warehouse.db") as conn:
            conn.execute("CREATE TABLE IF NOT EXISTS daily_revenue "
                         "(date TEXT, country TEXT, n_orders INT, revenue REAL)")
            conn.execute("DELETE FROM daily_revenue WHERE date = ?", (ds,))
            df.to_sql("daily_revenue", conn, if_exists="append", index=False)

    load(transform(extract()))


etl_demo()
'''

os.makedirs("../dags", exist_ok=True)
with open("../dags/etl_demo.py", "w") as f:
    f.write(DAG_CODE)
print("wrote ../dags/etl_demo.py — point AIRFLOW__CORE__DAGS_FOLDER at that dir and run `airflow standalone`")

## 6. Mini-project notes

- Add a `FileSensor` (or `@task.sensor`) gating extract on the source file existing.
- Trigger a failure (raise in transform) and watch the retry policy in the UI.
- Backfill: `airflow dags backfill etl_demo -s 2026-06-25 -e 2026-06-30`.
- Record here: what made each task idempotent, and what would break if it weren't.